In [162]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from sklearn.preprocessing import  OneHotEncoder, LabelEncoder
from sklearn.compose import make_column_transformer, make_column_selector
from tqdm import tqdm
from xgboost import XGBRegressor


In [163]:
housing=pd.read_csv("Housing.csv")
print("First five row")
print(housing.head())
print("\nShape of the Housing",housing.shape)
print("\nData type of the Housing\n",housing.dtypes)
print("\nMissing value counts",housing.isnull().sum())

First five row
     price  lotsize  bedrooms  bathrms  stories driveway recroom fullbase  \
0  42000.0     5850         3        1        2      yes      no      yes   
1  38500.0     4000         2        1        1      yes      no       no   
2  49500.0     3060         3        1        1      yes      no       no   
3  60500.0     6650         3        1        2      yes     yes       no   
4  61000.0     6360         2        1        1      yes      no       no   

  gashw airco  garagepl prefarea  
0    no    no         1       no  
1    no    no         0       no  
2    no    no         0       no  
3    no    no         0       no  
4    no    no         0       no  

Shape of the Housing (546, 12)

Data type of the Housing
 price       float64
lotsize       int64
bedrooms      int64
bathrms       int64
stories       int64
driveway        str
recroom         str
fullbase        str
gashw           str
airco           str
garagepl      int64
prefarea        str
dtype: object

In [164]:
X=housing.drop(columns='price')
y=housing['price']

In [165]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=25)


In [166]:
preprocess = make_column_transformer(
    (OneHotEncoder(drop="first", sparse_output=False), make_column_selector(dtype_include=['object', 'category'])),
    remainder="passthrough"
)

In [167]:
X_train_encoded = preprocess.fit_transform(X_train)
X_test_encoded = preprocess.transform(X_test)

In [168]:
from sklearn.metrics import r2_score
rates = [0.01, 0.05, 0.1]
n_estimators=[50,100]
max_depths=[3,5]
scores_XGB = []
for r in rates:
    for n in n_estimators:
        for d in max_depths:
            gbm=XGBRegressor(random_state=25,n_estimators=n,max_depth=d, learning_rate=r)
            gbm.fit(X_train_encoded,y_train)
            y_pred=gbm.predict(X_test_encoded)
        
            scores_XGB.append([r, n, d, r2_score(y_test, y_pred)])
           
            
            
            

In [169]:
df_scores_XGB = pd.DataFrame( scores_XGB, columns=['Learning Rate','Number of Estimators','Maximum Depth','R2 score'] )


In [170]:

print("R2 score from highest to lowest\n",df_scores_XGB.sort_values('R2 score',ascending=False))
highest_scoring_configuration=df_scores_XGB.sort_values('R2 score',ascending=False).iloc[0]
print("\nHighest-scoring configuration:")
print(highest_scoring_configuration)

R2 score from highest to lowest
     Learning Rate  Number of Estimators  Maximum Depth  R2 score
6            0.05                   100              3  0.619739
8            0.10                    50              3  0.616997
10           0.10                   100              3  0.602814
4            0.05                    50              3  0.596801
7            0.05                   100              5  0.588742
5            0.05                    50              5  0.582350
9            0.10                    50              5  0.570771
11           0.10                   100              5  0.560056
3            0.01                   100              5  0.488779
2            0.01                   100              3  0.450938
1            0.01                    50              5  0.340971
0            0.01                    50              3  0.303722

Highest-scoring configuration:
Learning Rate             0.050000
Number of Estimators    100.000000
Maximum Depth       

In [171]:
from lightgbm import LGBMRegressor 

rates = [0.01, 0.05, 0.1]
n_est = [50, 100, 200]
depths = [3, 5, -1]
scores_LGBMR = []
for r in tqdm(rates):
    for n in n_est:
        for d in depths:
            gbm = LGBMRegressor (random_state=25,n_estimators=n,verbose=-1, 
                                             max_depth=d, learning_rate=r)
            gbm.fit(X_train_encoded, y_train)
            y_pred = gbm.predict(X_test_encoded)
            scores_LGBMR.append( [r, n, d, r2_score(y_test, y_pred)] )


100%|██████████| 3/3 [00:00<00:00,  5.86it/s]


In [172]:
df_scores_LGBMR = pd.DataFrame( scores_LGBMR, columns=['Learning Rate','Number of Estimators','Maximum Depth','R2 score'] )
df_scores_LGBMR.sort_values( 'R2 score', ascending=False ) 

,Learning Rate,Number of Estimators,Maximum Depth,R2 score
14,0.05,100,-1,0.641652
20,0.10,50,-1,0.640175
18,0.10,50,3,0.635631
13,0.05,100,5,0.634652
12,0.05,100,3,0.633551
19,0.10,50,5,0.626121
11,0.05,50,-1,0.621275
17,0.05,200,-1,0.620178
21,0.10,100,3,0.620093
10,0.05,50,5,0.618426


In [173]:
highest_scoring_LightGBM_configuration=df_scores_LGBMR.sort_values( 'R2 score', ascending=False ).iloc[0]
print("\nHighest-scoring configuration:")
print(highest_scoring_LightGBM_configuration)


Highest-scoring configuration:
Learning Rate             0.050000
Number of Estimators    100.000000
Maximum Depth            -1.000000
R2 score                  0.641652
Name: 14, dtype: float64


In [174]:
from catboost import CatBoostRegressor

rates =[0.01, 0.05, 0.1]
n_est = [50, 100]
depths = [3, 5]
scores_catboost_encode = []
for r in tqdm(rates):
    for n in n_est:
        for d in depths:
            gbm = CatBoostRegressor(random_state=25,n_estimators=n,verbose=0, 
                                     max_depth=d, learning_rate=r)
            gbm.fit(X_train_encoded, y_train)
            y_pred_prob = gbm.predict(X_test_encoded)
            scores_catboost_encode.append( [r, n, d, r2_score(y_test, y_pred)] )


100%|██████████| 3/3 [00:02<00:00,  1.11it/s]


In [175]:
df_scores_catboost_encode = pd.DataFrame( scores_catboost_encode, columns=['Learning Rate','Number of Estimators','Maximum Depth','R2 score'] )
df_scores_catboost_encode.sort_values( 'R2 score', ascending=False )

,Learning Rate,Number of Estimators,Maximum Depth,R2 score
0,0.01,50,3,0.596065
1,0.01,50,5,0.596065
2,0.01,100,3,0.596065
3,0.01,100,5,0.596065
4,0.05,50,3,0.596065
5,0.05,50,5,0.596065
6,0.05,100,3,0.596065
7,0.05,100,5,0.596065
8,0.10,50,3,0.596065
9,0.10,50,5,0.596065


In [176]:
highest_scoring_CatBoost_withOneHotEncoder_configuration=df_scores_catboost_encode.sort_values( 'R2 score', ascending=False ).iloc[0]
print("\nHighest-scoring configuration:")
print(highest_scoring_CatBoost_withOneHotEncoder_configuration)


Highest-scoring configuration:
Learning Rate            0.010000
Number of Estimators    50.000000
Maximum Depth            3.000000
R2 score                 0.596065
Name: 0, dtype: float64


In [177]:
categorical_features=X_train.select_dtypes(include=['object','category']).columns.tolist()
print(categorical_features)

['driveway', 'recroom', 'fullbase', 'gashw', 'airco', 'prefarea']


C:\Users\HP\AppData\Local\Temp\ipykernel_13760\4110400563.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features=X_train.select_dtypes(include=['object','category']).columns.tolist()


In [178]:
baseline_cb=CatBoostRegressor(random_state=25,cat_features=categorical_features,verbose=0)
baseline_cb.fit(X_train,y_train)
y_pred_baseline=baseline_cb.predict(X_test)
baseline_r2_score=r2_score(y_test,y_pred_baseline)
print(f"Baseline Test R² Score: {baseline_r2_score}")


Baseline Test R² Score: 0.6158439632175752


In [179]:
rates =[0.01, 0.05, 0.1]
n_est = [50, 100]
depths = [3, 5]
scores_catboost = []
for r in tqdm(rates):
    for n in n_est:
        for d in depths:
            gbm = CatBoostRegressor(random_state=25,n_estimators=n,verbose=0,cat_features=categorical_features ,
                                     max_depth=d, learning_rate=r)
            gbm.fit(X_train, y_train)
            y_pred_prob = gbm.predict(X_test)
            scores_catboost.append( [r, n, d, r2_score(y_test, y_pred)] )

100%|██████████| 3/3 [00:02<00:00,  1.17it/s]


In [180]:
df_scores_catboost = pd.DataFrame(scores_catboost, columns=['Learning Rate', 'Number of Estimators', 'Maximum Depth', 'R2 score'])


df_scores_cb_sorted = df_scores_catboost.sort_values(by='R2 score', ascending=False)
print(df_scores_cb_sorted)

    Learning Rate  Number of Estimators  Maximum Depth  R2 score
0            0.01                    50              3  0.596065
1            0.01                    50              5  0.596065
2            0.01                   100              3  0.596065
3            0.01                   100              5  0.596065
4            0.05                    50              3  0.596065
5            0.05                    50              5  0.596065
6            0.05                   100              3  0.596065
7            0.05                   100              5  0.596065
8            0.10                    50              3  0.596065
9            0.10                    50              5  0.596065
10           0.10                   100              3  0.596065
11           0.10                   100              5  0.596065


In [181]:
best_xgb=df_scores_XGB.sort_values(by='R2 score',ascending=False).iloc[0]
best_lgbmr=df_scores_LGBMR.sort_values(by='R2 score',ascending=False).iloc[0]
best_catboost_encode=df_scores_catboost_encode.sort_values(by='R2 score',ascending=False).iloc[0]
best_catboost=df_scores_catboost.sort_values(by='R2 score',ascending=False).iloc[0]

summary_data = [
    ['XGBoost', best_xgb['Learning Rate'], best_xgb['Number of Estimators'], best_xgb['Maximum Depth'], best_xgb['R2 score']],
    ['LightGBM', best_lgbmr['Learning Rate'], best_lgbmr['Number of Estimators'], best_lgbmr['Maximum Depth'], best_lgbmr['R2 score']],
    ['CatBoost with One-Hot Encoding', best_catboost_encode['Learning Rate'], best_catboost_encode['Number of Estimators'], best_catboost_encode['Maximum Depth'], best_catboost_encode['R2 score']],
    ['CatBoost without One-Hot Encoding', best_catboost['Learning Rate'], best_catboost['Number of Estimators'], best_catboost['Maximum Depth'], best_catboost['R2 score']]
]

df_summary = pd.DataFrame(summary_data, columns=['Experiment', 'Learning Rate', 'Number of Estimators', 'Maximum Depth', 'R²'])

print(df_summary)

                          Experiment  Learning Rate  Number of Estimators  \
0                            XGBoost           0.05                 100.0   
1                           LightGBM           0.05                 100.0   
2     CatBoost with One-Hot Encoding           0.01                  50.0   
3  CatBoost without One-Hot Encoding           0.01                  50.0   

   Maximum Depth        R²  
0            3.0  0.619739  
1           -1.0  0.641652  
2            3.0  0.596065  
3            3.0  0.596065  


Top Performing Model: LightGBM achieved the highest test R2 score of 0.6417 using a learning rate of 0.05, 100 estimators, and an unconstrained maximum depth (-1).

Comparison: XGBoost performed second best with an R2 score of 0.6197 (learning rate 0.05, 100 estimators, depth 3), while both CatBoost experiments (with and without One-Hot Encoding) produce identical scores of 0.5961 (learning rate 0.01, 50 estimators, depth 3).

Conclusion: LightGBM proved to be the most effective algorithm for this housing dataset, successfully leveraging tree depth and moderate learning rates to capture more complex patterns than XGBoost and CatBoost.